# 05 · The methods: plan check, tree, segments, stress test

**Produces:** the numbers behind our claim, each with its denominator and period: `city_month_check.csv`, `tree_network.csv`, `tree_by_city.csv`, `segments_channel.csv`, `segments_source.csv`, `gap_*.csv`, `tree_boxes.csv`, `sensitivity_table.csv`, `recording_check.csv`.

**Needs:** notebook 03 run (and notebook 02 for the stress test).

**The question:** Dr Menon asked which part of the business is short, by how much, and where. We answer with (1) the plan check, (2) the tree *tests = people × visits per person × tests per visit*, (3) the split by city, channel and source, and (4) an attempt to break our own answer.

## CONFIG
Everything that depends on the machine or the period lives in this one cell, so nothing else needs editing.

In [ ]:
# ---------- imports ----------
import gzip
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# show more columns when we print a dataframe
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

# ---------- paths ----------
# the notebooks live in the notebooks folder, so the repo root is one folder up
if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

RAW = ROOT / "data" / "raw"       # the data pack as we received it (we never edit this)
CLEAN = ROOT / "data" / "clean"   # the files our notebooks write
SLIDES = ROOT / "slides"          # charts and the excel file for the slides
CLEAN.mkdir(parents=True, exist_ok=True)
SLIDES.mkdir(parents=True, exist_ok=True)

# ---------- the two periods we compare ----------
H1_2025 = ("2025-01-01", "2025-06-30 23:59:59")   # 23:59:59 so that 30 June is included
H1_2026 = ("2026-01-01", "2026-06-30 23:59:59")

# a small helper: prints PASS or FAIL but never stops the notebook
def check(name, condition, detail=""):
    if condition:
        print("PASS", name, detail)
    else:
        print("FAIL", name, detail)

print("repo root:", ROOT)

In [ ]:
# ---------- load the clean table from notebook 03 ----------
df = pd.read_csv(CLEAN / "bookings_clean.csv.gz", parse_dates=["slot_at"],
                 dtype={"booking_id": "str", "patient_id": "str", "person_id": "str", "test_code": "str"}, low_memory=False)
print(df.shape)
print(df.dtypes)

# which half-year does every row belong to? we use slot_at, never the file name
df["half"] = "other"
df.loc[df["slot_at"].between(H1_2025[0], H1_2025[1]), "half"] = "H1_2025"
df.loc[df["slot_at"].between(H1_2026[0], H1_2026[1]), "half"] = "H1_2026"
df["month"] = df["slot_at"].dt.strftime("%Y-%m")
df["quarter"] = df["slot_at"].dt.to_period("Q").astype(str)

df_h = df[df["half"] != "other"].copy()       # only the two halves we compare
df_h["all"] = "Network"
print(df_h["half"].value_counts())

total_2025 = (df_h["half"] == "H1_2025").sum()
total_2026 = (df_h["half"] == "H1_2026").sum()
print("network tests: H1 2025 =", total_2025, " H1 2026 =", total_2026, " growth =", round((total_2026 / total_2025 - 1) * 100, 2), "%")

# the plan, from wide (one column per month) to long (one row per city and month)
plan_wide = pd.read_csv(RAW / "plan" / "plan_2026_H1.csv")
plan_long = plan_wide.melt(id_vars=["city", "metric"], var_name="month", value_name="plan_2026")
plan_long["city"] = plan_long["city"].replace({"Bangalore": "Bengaluru"})
plan_long = plan_long[["city", "month", "plan_2026"]]
plan_city = plan_long.groupby("city")["plan_2026"].sum()
plan_total = plan_city.sum()
print("plan total:", plan_total, " plan growth:", round((plan_total / total_2025 - 1) * 100, 2), "%")

## 1 · Plan check: city by month
The plan was set as last year's count times a growth factor. If our counting rule matches the plan's, then `plan ÷ 2025 count` should be **steady within each city** (it will not be exactly 1.18 everywhere: each city has its own factor).

In [ ]:
# tests per city per month
df_counts = df.groupby(["city", "month"]).size().reset_index(name="tests")

# 2025 months are renamed to 2026, so that January 2025 sits next to the January 2026 plan
df_c25 = df_counts[df_counts["month"].str.startswith("2025")].copy()
df_c25["month"] = df_c25["month"].str.replace("2025", "2026", regex=False)
df_c25 = df_c25.rename(columns={"tests": "actual_2025"})
df_c26 = df_counts[df_counts["month"].str.startswith("2026")].rename(columns={"tests": "actual_2026"})

# LEFT join from the plan, so a missing month shows as empty instead of disappearing
df_cm = plan_long.merge(df_c25, on=["city", "month"], how="left").merge(df_c26, on=["city", "month"], how="left")
check("the join kept every plan row", len(df_cm) == len(plan_long), str(len(plan_long)) + " rows")

df_cm["plan_over_2025"] = df_cm["plan_2026"] / df_cm["actual_2025"]
df_cm["growth_pct"] = (df_cm["actual_2026"] / df_cm["actual_2025"] - 1) * 100
df_cm["vs_plan_pct"] = (df_cm["actual_2026"] / df_cm["plan_2026"] - 1) * 100
df_cm.to_csv(CLEAN / "city_month_check.csv", index=False)

print("CHECK 1: plan / 2025 count (steady inside a city means our rule matches the plan)")
print(df_cm.pivot(index="month", columns="city", values="plan_over_2025").round(3))
print()
print("CHECK 2: growth of 2026 against the same month of 2025, %")
print(df_cm.pivot(index="month", columns="city", values="growth_pct").round(1))

In [ ]:
# half-year by city, and the gap against the plan in POINTS
# gap points = (actual 2026 - plan) / network tests in H1 2025 * 100.  every city uses the same denominator, so the six numbers add up
df_city = df_cm.groupby("city")[["actual_2025", "actual_2026", "plan_2026"]].sum()
df_city["growth_pct"] = (df_city["actual_2026"] / df_city["actual_2025"] - 1) * 100
df_city["plan_growth_pct"] = (df_city["plan_2026"] / df_city["actual_2025"] - 1) * 100
df_city["gap_points"] = (df_city["actual_2026"] - df_city["plan_2026"]) / df_city["actual_2025"].sum() * 100
print(df_city.round(1))

n25 = df_city["actual_2025"].sum()
n26 = df_city["actual_2026"].sum()
print()
print("NETWORK: H1 2025 =", int(n25), " H1 2026 =", int(n26), " plan =", int(plan_total))
print("growth =", round((n26 / n25 - 1) * 100, 1), "%   plan growth =", round((plan_total / n25 - 1) * 100, 1), "%   gap =", round(df_city["gap_points"].sum(), 1), "points =", int(plan_total - n26), "tests")
check("city gap points add up to the network gap", abs(df_city["gap_points"].sum() - (n26 - plan_total) / n25 * 100) < 0.001)

## 2 · The tree: tests = people × visits per person × tests per visit
- **tests** = rows (one per booking-test)
- **visits** = distinct booking ids
- **people** = distinct person ids (patient ids merged in notebook 03)

We use three branches and not two (people × tests) because the middle branch separates *fewer people came* from *the same people came less often*. Those need different fixes.

In [ ]:
def tree_table(df_in, group_cols):
    grouped = df_in.groupby(group_cols + ["half"]).agg(tests=("booking_id", "size"),
                                                       visits=("booking_id", "nunique"),
                                                       people=("person_id", "nunique"))
    grouped["visits_per_person"] = grouped["visits"] / grouped["people"]
    grouped["tests_per_visit"] = grouped["tests"] / grouped["visits"]
    return grouped

def tree_growth(t):
    # growth of every box in %, and how many POINTS of the total growth each branch carries (log split: the three add up)
    a = t.xs("H1_2025", level="half")
    b = t.xs("H1_2026", level="half")
    g = (b / a - 1) * 100
    total_log = np.log(b["tests"] / a["tests"])
    for col in ["people", "visits_per_person", "tests_per_visit"]:
        g[col + "_pts"] = np.log(b[col] / a[col]) / total_log * g["tests"]
    return g

tree_net = tree_table(df_h, ["all"])
print(tree_net.round(3))
tree_net.to_csv(CLEAN / "tree_network.csv")

g_net = tree_growth(tree_net).iloc[0]
print()
print("growth %: tests", round(g_net["tests"], 2), "| people", round(g_net["people"], 2),
      "| visits per person", round(g_net["visits_per_person"], 2), "| tests per visit", round(g_net["tests_per_visit"], 2))

In [ ]:
# CHECK: the three branches must multiply back to the total
t25 = tree_net.loc[("Network", "H1_2025")]
t26 = tree_net.loc[("Network", "H1_2026")]
product = (t26["people"] / t25["people"]) * (t26["visits_per_person"] / t25["visits_per_person"]) * (t26["tests_per_visit"] / t25["tests_per_visit"])
check("branches multiply back to the total", abs(product - t26["tests"] / t25["tests"]) < 1e-9,
      "product " + str(round((product - 1) * 100, 3)) + "% vs tests " + str(round((t26["tests"] / t25["tests"] - 1) * 100, 3)) + "%")

pts = g_net["people_pts"] + g_net["visits_per_person_pts"] + g_net["tests_per_visit_pts"]
check("branch points add up to the total growth", abs(pts - g_net["tests"]) < 1e-6)
print("points carried: people", round(g_net["people_pts"], 2), " visits per person", round(g_net["visits_per_person_pts"], 2),
      " tests per visit", round(g_net["tests_per_visit_pts"], 2))

# the weakest branch = the one with the lowest growth
branches = {"people": g_net["people"], "visits per person": g_net["visits_per_person"], "tests per visit": g_net["tests_per_visit"]}
weakest = min(branches, key=branches.get)
print("weakest branch:", weakest)

In [ ]:
# the same tree for every city
tree_city = tree_table(df_h, ["city"])
g_city = tree_growth(tree_city)
g_city["gap_points"] = df_city["gap_points"]
print(g_city[["tests", "people", "visits_per_person", "tests_per_visit", "gap_points"]].round(1))
g_city.to_csv(CLEAN / "tree_by_city.csv")

> **Interview angle.** Revenue grew 10 percent and patients grew 4. Decompose the rest, and say what you would check before presenting it.
>
> *A good answer:* about 6 points sit in visits per patient and revenue per visit together (the log split makes it exact). Before presenting: check the patient count is people and not ids, that both periods use the same definition of a visit, and that a system change did not move the grain.

## 3 · Where is it short? City, channel and source
The plan has only cities. To split the plan over channels and sources we **spread each city's plan in last year's mix** (an assumption: the plan has no channel or source split). Growth percentages need no assumption.

In [ ]:
def seg_table(dim):
    g = df_h.groupby([dim, "half"]).agg(tests=("booking_id", "size"), visits=("booking_id", "nunique"),
                                        people=("person_id", "nunique")).unstack("half").fillna(0)
    out = pd.DataFrame(index=g.index)
    out["tests_2025"] = g[("tests", "H1_2025")]
    out["tests_2026"] = g[("tests", "H1_2026")]
    out["growth_pct"] = (out["tests_2026"] / out["tests_2025"] - 1) * 100
    out["share_2025_pct"] = out["tests_2025"] / total_2025 * 100
    out["share_2026_pct"] = out["tests_2026"] / total_2026 * 100
    out["pts_of_network_growth"] = (out["tests_2026"] - out["tests_2025"]) / total_2025 * 100     # these add up to the network growth
    out["people_pct"] = (g[("people", "H1_2026")] / g[("people", "H1_2025")] - 1) * 100
    vpp_25 = g[("visits", "H1_2025")] / g[("people", "H1_2025")]
    vpp_26 = g[("visits", "H1_2026")] / g[("people", "H1_2026")]
    out["visits_per_person_pct"] = (vpp_26 / vpp_25 - 1) * 100
    tpv_25 = g[("tests", "H1_2025")] / g[("visits", "H1_2025")]
    tpv_26 = g[("tests", "H1_2026")] / g[("visits", "H1_2026")]
    out["tests_per_visit_pct"] = (tpv_26 / tpv_25 - 1) * 100
    return out.round(2)

for dim in ["channel", "source"]:
    seg = seg_table(dim)
    print("BY", dim.upper())
    print(seg.to_string())
    network_growth = (total_2026 / total_2025 - 1) * 100
    check("points add up to the network growth (" + dim + ")", abs(seg["pts_of_network_growth"].sum() - network_growth) < 0.05)
    seg.to_csv(CLEAN / ("segments_" + dim + ".csv"))
    print()

In [ ]:
# gap against the plan, with each city's plan spread over its segments in last year's mix
def gap_allocation(dims):
    cols = ["city"] + [d for d in dims if d != "city"]
    x = df_h.groupby(cols + ["half"]).size().unstack("half").fillna(0)
    city_2025 = x.groupby(level="city")["H1_2025"].transform("sum").values
    city_plan = np.array(x.index.get_level_values("city").map(plan_city))
    x["plan_alloc"] = city_plan * x["H1_2025"].values / city_2025
    x["gap_pts"] = (x["H1_2026"] - x["plan_alloc"]) / total_2025 * 100
    out = x.groupby(level=dims).agg(tests_2025=("H1_2025", "sum"), tests_2026=("H1_2026", "sum"),
                                    plan_alloc=("plan_alloc", "sum"), gap_pts=("gap_pts", "sum"))
    out["growth_pct"] = (out["tests_2026"] / out["tests_2025"] - 1) * 100
    return out.round(2)

gap_channel = gap_allocation(["channel"])
gap_source = gap_allocation(["source"])
gap_city_channel = gap_allocation(["city", "channel"])
print("GAP BY CHANNEL (points)"); print(gap_channel)
print(); print("GAP BY SOURCE (points)"); print(gap_source)
print(); print("GAP BY CITY x CHANNEL (points), worst first"); print(gap_city_channel.sort_values("gap_pts").head(8))
check("channel gaps add up to the network gap", abs(gap_channel["gap_pts"].sum() - (total_2026 - plan_total) / total_2025 * 100) < 0.05)
for name, tab in [("channel", gap_channel), ("source", gap_source), ("city_channel", gap_city_channel)]:
    tab.to_csv(CLEAN / ("gap_" + name + ".csv"))

In [ ]:
# is clinic below its (allocated) plan in every city, and home at or above it?
clinic_gaps = gap_city_channel.xs("clinic", level="channel")["gap_pts"]
home_gaps = gap_city_channel.xs("home", level="channel")["gap_pts"]
print("clinic gap points by city:")
print(clinic_gaps)
print("home gap points by city:")
print(home_gaps)
clinic_below_everywhere = bool((clinic_gaps < 0).all())
print("clinic below plan in every city:", clinic_below_everywhere)

### Time: month, quarter, half-year (did the shortfall start in Q2? did the offer change anything?)

In [ ]:
# year on year by quarter, for the network, the channels and the sources
def yoy_quarters(dim):
    t = df.groupby([dim, "quarter"]).size().unstack("quarter")
    q1 = (t["2026Q1"] / t["2025Q1"] - 1) * 100
    q2 = (t["2026Q2"] / t["2025Q2"] - 1) * 100
    return pd.DataFrame({"Q1_yoy_pct": q1, "Q2_yoy_pct": q2}).round(1)

df["all"] = "Network"
df_q = pd.concat([yoy_quarters("all"), yoy_quarters("channel"), yoy_quarters("source")])
print("YEAR ON YEAR BY QUARTER (the offer ran in Q1)")
print(df_q)

# month by month for the network and the two channels
m_all = df.groupby("month").size()
m_clinic = df[df["channel"] == "clinic"].groupby("month").size()
m_home = df[df["channel"] == "home"].groupby("month").size()
rows = []
for i in range(1, 7):
    m26 = "2026-0" + str(i)
    m25 = "2025-0" + str(i)
    rows.append({"month": m26,
                 "network": (m_all[m26] / m_all[m25] - 1) * 100,
                 "clinic": (m_clinic[m26] / m_clinic[m25] - 1) * 100,
                 "home": (m_home[m26] / m_home[m25] - 1) * 100})
df_monthly = pd.DataFrame(rows).set_index("month").round(1)
print()
print("MONTHLY YEAR ON YEAR %")
print(df_monthly)
df_monthly.to_csv(CLEAN / "monthly_yoy_channel.csv")

In [ ]:
# half-year sequence: uses the WHOLE table (df), not df_h, so H2 2025 is there
# (this is the check that the plan assumed a flat +18% on every month: seasonality)
df["half_full"] = "other"
df.loc[df["slot_at"].between("2025-01-01", "2025-06-30 23:59:59"), "half_full"] = "H1_2025"
df.loc[df["slot_at"].between("2025-07-01", "2025-12-31 23:59:59"), "half_full"] = "H2_2025"
df.loc[df["slot_at"].between("2026-01-01", "2026-06-30 23:59:59"), "half_full"] = "H1_2026"
seq = df[df["half_full"] != "other"].groupby(["channel", "half_full"]).size().unstack("half_full")
seq = seq[["H1_2025", "H2_2025", "H1_2026"]]
seq.loc["ALL"] = seq.sum()
seq["H2_vs_H1_2025_pct"] = (seq["H2_2025"] / seq["H1_2025"] - 1) * 100
seq["H1_2026_vs_H2_2025_pct"] = (seq["H1_2026"] / seq["H2_2025"] - 1) * 100
print(seq.round(1))

# packages: a package is ONE row, so more packages can change tests per visit without fewer patients
df_h["is_package"] = df_h["test_code"].fillna("").str.startswith("PK")
pk = df_h.groupby(["channel", "half"]).agg(package_share_pct=("is_package", "mean"), rows=("booking_id", "size"), visits=("booking_id", "nunique"))
pk["package_share_pct"] = pk["package_share_pct"] * 100
pk["tests_per_visit"] = pk["rows"] / pk["visits"]
print()
print(pk[["package_share_pct", "tests_per_visit"]].round(2))

## 4 · Fill every box of the handbook's trees
Brief A tree: *Test volume = Patients served × Visits per patient × Tests per visit*, *Patients served = New + Returning*, *Revenue = Test volume × Revenue per test*. Revenue here is **booked at list price** (`test_price`), not billed.

In [ ]:
rows = []
for half_name in ["H1_2025", "H1_2026"]:
    d = df_h[df_h["half"] == half_name]
    n_tests = len(d)
    n_visits = d["booking_id"].nunique()
    n_people = d["person_id"].nunique()
    revenue = d["test_price"].sum()
    rows.append({"half": half_name, "patients_served": n_people, "visits": n_visits, "tests": n_tests,
                 "visits_per_patient": n_visits / n_people, "tests_per_visit": n_tests / n_visits,
                 "tests_per_patient": n_tests / n_people, "revenue_Rs_crore": revenue / 10000000,
                 "revenue_per_test_Rs": revenue / n_tests})
df_boxes = pd.DataFrame(rows).set_index("half").T
df_boxes["change_pct"] = (df_boxes["H1_2026"] / df_boxes["H1_2025"] - 1) * 100
print(df_boxes.round(3))

for half_name in ["H1_2025", "H1_2026"]:
    b = df_boxes[half_name]
    check(half_name + ": tests = patients x visits per patient x tests per visit", abs(b["patients_served"] * b["visits_per_patient"] * b["tests_per_visit"] - b["tests"]) < 1e-6 * b["tests"])
    check(half_name + ": revenue = tests x revenue per test", abs(b["tests"] * b["revenue_per_test_Rs"] / 10000000 - b["revenue_Rs_crore"]) < 1e-6 * b["revenue_Rs_crore"])

In [ ]:
# new against returning patients: a patient is NEW in a half if they signed up inside it
# (the data starts in January 2025, so "seen before" alone cannot split H1 2025: we use signup_date from the patient master)
df_pat = pd.read_csv(RAW / "patients" / "patients.csv.gz", dtype=str, usecols=["patient_id", "signup_date"])
df_pat["signup"] = pd.to_datetime(df_pat["signup_date"], errors="coerce")
if df_pat["signup"].isnull().mean() > 0.05:
    # many dates could not be read: they may be written day first
    df_pat["signup"] = pd.to_datetime(df_pat["signup_date"], errors="coerce", dayfirst=True)
print("signup dates that could not be read:", round(df_pat["signup"].isnull().mean() * 100, 1), "%")

# one signup date per PERSON: the earliest of the person's patient ids
person_map = pd.read_csv(CLEAN / "person_map.csv.gz", dtype=str)
df_pat = df_pat.merge(person_map, on="patient_id", how="left")
df_pat["person_id"] = df_pat["person_id"].fillna(df_pat["patient_id"])
signup_by_person = df_pat.groupby("person_id")["signup"].min()

rows = []
for half_name, period in [("H1_2025", H1_2025), ("H1_2026", H1_2026)]:
    people_in_half = df_h[df_h["half"] == half_name]["person_id"].unique()
    signups = signup_by_person.reindex(people_in_half)
    start_of_half = pd.Timestamp(period[0])
    rows.append({"half": half_name, "new_patients": int((signups >= start_of_half).sum()),
                 "returning_patients": int((signups < start_of_half).sum()), "signup_unknown": int(signups.isnull().sum()),
                 "patients_served": len(people_in_half)})
df_newret = pd.DataFrame(rows).set_index("half")
print(df_newret)
check("new + returning + unknown = patients served", (df_newret["new_patients"] + df_newret["returning_patients"] + df_newret["signup_unknown"] == df_newret["patients_served"]).all())
df_boxes.to_csv(CLEAN / "tree_boxes.csv")
df_newret.to_csv(CLEAN / "tree_new_returning.csv")

## 5 · Try to break our own claim
Three questions: how fragile is the headline (sensitivity), is any fall a counting artefact (recording check), and what would it take to reach the plan (what-if).

In [ ]:
# ---------- we need the table BEFORE cleaning, to try the alternative decisions ----------
df_all = pd.read_csv(CLEAN / "bookings_all.csv.gz", parse_dates=["slot_at"],
                     dtype={"booking_id": "str", "status_at_export": "str", "test_code": "str"}, low_memory=False)

coll_list = []
for path in sorted((RAW / "collections").glob("*.csv.gz")):
    coll_list.append(pd.read_csv(path, dtype=str))
df_coll = pd.concat(coll_list, ignore_index=True)
df_coll["booking_id"] = df_coll["booking_id"].str.strip()
df_coll = df_coll.drop_duplicates().drop_duplicates(subset="booking_id")

df_all = df_all.merge(df_coll[["booking_id", "status"]].rename(columns={"status": "coll_status"}), on="booking_id", how="left", validate="many_to_one")
df_all["coll_status"] = df_all["coll_status"].fillna("no_collection_row")
print(df_all.shape)

def make_variant(keep_old_copies=False, keep_empty=False, rule="A", drop_blank_code=False, only_cities=None):
    d = df_all
    if not keep_old_copies:
        d = d[~d["status_at_export"].eq("rescheduled").fillna(False).astype(bool)]
    if not keep_empty:
        d = d[~((d["system"] == "nova") & d["test_code"].isnull() & d["test_price"].isnull())]
    if rule == "B":
        d = d[d["coll_status"] == "collected"]
    if rule == "C":
        d = d[d["coll_status"].isin(["collected", "no_show"])]
    if drop_blank_code:
        d = d[d["test_code"].notnull()]
    if only_cities is not None:
        d = d[d["city"].isin(only_cities)]
    return d

def growth_of(d, unit="tests"):
    in_2025 = d[d["slot_at"].between(H1_2025[0], H1_2025[1])]
    in_2026 = d[d["slot_at"].between(H1_2026[0], H1_2026[1])]
    if unit == "tests":
        a, b = len(in_2025), len(in_2026)
    else:
        a, b = in_2025["booking_id"].nunique(), in_2026["booking_id"].nunique()
    return a, b, (b / a - 1) * 100

In [ ]:
# ---------- the sensitivity table ----------
four_cities = ["Ahmedabad", "Bengaluru", "Hyderabad", "Pune"]     # the cities that never changed booking system
variants = [
    ("BASELINE (our decisions)", make_variant(), "tests"),
    ("keep nova rescheduled originals", make_variant(keep_old_copies=True), "tests"),
    ("keep nova empty-test placeholders", make_variant(keep_empty=True), "tests"),
    ("count rule A: all rows", make_variant(rule="A"), "tests"),
    ("count rule B: collected only", make_variant(rule="B"), "tests"),
    ("count rule C: collected + no-show", make_variant(rule="C"), "tests"),
    ("drop rows with blank test_code", make_variant(drop_blank_code=True), "tests"),
    ("only the 4 cities that never changed system", make_variant(only_cities=four_cities), "tests"),
    ("count visits (bookings), not tests", make_variant(), "bookings"),
]
rows = []
for name, d, unit in variants:
    a, b, g = growth_of(d, unit)
    rows.append({"variant": name, "H1_2025": a, "H1_2026": b, "growth_pct": round(g, 2)})
df_sens = pd.DataFrame(rows)
df_sens["moved_vs_baseline_pts"] = (df_sens["growth_pct"] - df_sens["growth_pct"].iloc[0]).round(2)
print(df_sens.to_string(index=False))
df_sens.to_csv(CLEAN / "sensitivity_table.csv", index=False)
print("the 3 rows with the biggest moved_vs_baseline_pts go on slide 5")

In [ ]:
# ---------- recording check ----------
# the collection app did NOT change system in April. the booking system did (Chennai and Jaipur).
# if a city's BOOKINGS fall but its COLLECTED samples do not, the fall is a counting artefact.
CODE_TO_CITY = {"AHM": "Ahmedabad", "BLR": "Bengaluru", "CHE": "Chennai", "HYD": "Hyderabad", "JAI": "Jaipur", "PUN": "Pune"}

base = make_variant()
base = base.assign(quarter=base["slot_at"].dt.to_period("Q").astype(str))
bookings_q = base.groupby(["city", "quarter"])["booking_id"].nunique().unstack("quarter")

df_coll["city"] = df_coll["clinic_id"].str[:3].map(CODE_TO_CITY)
df_coll["collected_dt"] = pd.to_datetime(df_coll["collected_at"], errors="coerce")
df_coll["quarter"] = df_coll["collected_dt"].dt.to_period("Q").astype(str)
collected_q = df_coll[df_coll["status"] == "collected"].groupby(["city", "quarter"]).size().unstack("quarter")

df_rec = pd.DataFrame({
    "bookings_Q1": (bookings_q["2026Q1"] / bookings_q["2025Q1"] - 1) * 100,
    "bookings_Q2": (bookings_q["2026Q2"] / bookings_q["2025Q2"] - 1) * 100,
    "collected_Q1": (collected_q["2026Q1"] / collected_q["2025Q1"] - 1) * 100,
    "collected_Q2": (collected_q["2026Q2"] / collected_q["2025Q2"] - 1) * 100}).round(1)
df_rec["bookings_change_Q1_to_Q2"] = (df_rec["bookings_Q2"] - df_rec["bookings_Q1"]).round(1)
df_rec["collected_change_Q1_to_Q2"] = (df_rec["collected_Q2"] - df_rec["collected_Q1"]).round(1)
print("YEAR ON YEAR %, by city and quarter (bookings = system that changed, collected = independent app)")
print(df_rec)
print("READ IT: if a city's bookings growth drops from Q1 to Q2 but its collected growth does not, the drop is a counting artefact.")
df_rec.to_csv(CLEAN / "recording_check.csv")

In [ ]:
# ---------- the ten-minute check: does the independent collections app show the same clinic / home shift? ----------
df_coll["half"] = "other"
df_coll.loc[df_coll["collected_dt"].between(H1_2025[0], H1_2025[1]), "half"] = "H1_2025"
df_coll.loc[df_coll["collected_dt"].between(H1_2026[0], H1_2026[1]), "half"] = "H1_2026"
col_ch = df_coll[(df_coll["status"] == "collected") & (df_coll["half"] != "other")].groupby(["channel", "half"]).size().unstack("half")
col_ch["growth_pct"] = (col_ch["H1_2026"] / col_ch["H1_2025"] - 1) * 100
print("COLLECTED SAMPLES by channel (from the collection app, not the booking system)")
print(col_ch.round(1))
print("if clinic falls and home rises here too, the channel shift is real and not a booking-system artefact")

# ---------- what-if ----------
f_people = t26["people"] / t25["people"]
f_visits = t26["visits_per_person"] / t25["visits_per_person"]
f_tests = t26["tests_per_visit"] / t25["tests_per_visit"]
factors = {"people": f_people, "visits per person": f_visits, "tests per visit": f_tests}
weak = min(factors, key=factors.get)
others = 1.0
for name in factors:
    if name != weak:
        others = others * factors[name]
needed = (plan_total / t25["tests"]) / others
print()
print("WHAT-IF: the weakest branch is", weak, "and it grew", round((factors[weak] - 1) * 100, 1), "%")
print("to reach the plan with the other two branches as observed it needed", round((needed - 1) * 100, 1), "%, which is", int(plan_total - t26["tests"]), "more tests")
print("ASSUMPTION to say on the slide: the other two branches stay exactly as observed")

## 6 · Link to Brief E: the free home collection offer (runs only if the campaign files are in the pack)
The offer waived the ₹150 home collection fee from January to March 2026. We ask: what does the offer look like in the data, who used it, and what did it cost per extra booking? There is no control group, so none of this proves cause.

In [ ]:
camp_path = RAW / "campaigns" / "campaign_bookings.csv.gz"
camp_info_path = RAW / "campaigns" / "campaigns.csv"
if camp_path.exists() and camp_info_path.exists():
    df_camp_info = pd.read_csv(camp_info_path)
    print(df_camp_info.to_string(index=False))
    df_tags = pd.read_csv(camp_path, dtype=str)
    df_tags["booking_id"] = df_tags["booking_id"].str.strip()
    print("tagged booking ids:", df_tags["booking_id"].nunique())

    # the offer window: first three months of 2026 (change here if campaigns.csv says otherwise)
    offer_start = pd.Timestamp("2026-01-01")
    offer_end = pd.Timestamp("2026-03-31 23:59:59")

    # one row per booking
    df_bk = df.groupby("booking_id").agg(slot_at=("slot_at", "first"), channel=("channel", "first"), person_id=("person_id", "first"),
                                         n_tests=("booking_id", "size"), value=("test_price", "sum"), home_fee=("home_fee", "first"))
    df_bk["offer"] = df_bk.index.isin(df_tags["booking_id"])
    df_bk["in_window"] = df_bk["slot_at"].between(offer_start, offer_end)
    tagged = df_bk[df_bk["offer"]]
    print("tagged found in bookings:", len(tagged), " tagged by channel:", tagged["channel"].value_counts().to_dict(), " tagged outside window:", int((~tagged["in_window"]).sum()))
    home_in_window = df_bk[(df_bk["channel"] == "home") & df_bk["in_window"]]
    print("home bookings in the window:", len(home_in_window), " share tagged:", round(home_in_window["offer"].mean() * 100, 1), "%")
    print("home fee = 0 on", round((home_in_window["home_fee"] == 0).mean() * 100, 1), "% of them")
    fees_waived = int(((df_bk["channel"] == "home") & df_bk["in_window"] & (df_bk["home_fee"] == 0)).sum()) * 150
    print("fees waived: about Rs", fees_waived)
else:
    print("no campaign files in data/raw/campaigns, skipping this section")

In [ ]:
if camp_path.exists() and camp_info_path.exists():
    # who used the offer? what did they do BEFORE the window?
    before = df_bk[df_bk["slot_at"] < offer_start]
    n_clinic_before = before[before["channel"] == "clinic"].groupby("person_id").size()
    n_home_before = before[before["channel"] == "home"].groupby("person_id").size()

    def behaviour_before(person_ids):
        shares = {"booked HOME before": 0, "booked CLINIC only before": 0, "no earlier booking in the data": 0}
        for p in person_ids:
            if p in n_home_before.index:
                shares["booked HOME before"] = shares["booked HOME before"] + 1
            elif p in n_clinic_before.index:
                shares["booked CLINIC only before"] = shares["booked CLINIC only before"] + 1
            else:
                shares["no earlier booking in the data"] = shares["no earlier booking in the data"] + 1
        total = len(person_ids)
        for key in shares:
            shares[key] = round(shares[key] / total * 100, 1)
        return shares

    users = tagged["person_id"].unique()
    others = df_bk[df_bk["in_window"] & ~df_bk["offer"] & ~df_bk["person_id"].isin(users)]["person_id"].unique()
    print("OFFER USERS (", len(users), "people ) % by behaviour before the window:", behaviour_before(users))
    print("OTHERS      (", len(others), "people ) % by behaviour before the window:", behaviour_before(others))
    print("many CLINIC-only before = the offer moved existing clinic patients to home (switching). many HOME before = they would have booked anyway.")

    # marketing's comparison against the same quarter last year
    q_counts = df_bk.groupby(df_bk["slot_at"].dt.to_period("Q").astype(str)).size()
    print()
    print("marketing's comparison, Q1 2026 vs Q4 2025:", round((q_counts["2026Q1"] / q_counts["2025Q4"] - 1) * 100, 1), "%")
    y1 = (q_counts["2026Q1"] / q_counts["2025Q1"] - 1) * 100
    y2 = (q_counts["2026Q2"] / q_counts["2025Q2"] - 1) * 100
    print("same quarter last year: Q1", round(y1, 1), "%  Q2", round(y2, 1), "%  -> the offer quarter ran", round(y1 - y2, 1), "points above the quarter after")
    extra_bookings = (y1 - y2) / 100 * q_counts["2025Q1"]
    if extra_bookings > 0:
        print("waived fees Rs", fees_waived, "/", int(extra_bookings), "extra bookings = Rs", int(fees_waived / extra_bookings), "per extra booking, against an average booking worth Rs", int(df_bk[df_bk["in_window"]]["value"].mean()), "at list price")
        print("this is an order-of-magnitude estimate: there is no control group and Q2 is not a clean no-offer baseline")

## SUM
- The headline, the tree and the splits are saved as csv files in `data/clean/`.
- The three rows with the biggest `moved_vs_baseline_pts` go on slide 5 and into the approach log.
- Any gap number by channel or source must carry the sentence "the plan was spread over channels in last year's mix".

## Before you move on
- Which number is on slide 3, and what are its denominator and period? Say them aloud.
- Which of the methods did we use, and why not the nearest other one? (an approach-log row each)
- What would make us wrong? (slide 5)